# Confined Space Entry Permit — OCR Key-Value Extractor

**Purpose:** Extract structured key-value pairs from scanned Confined Space Entry Permit forms (OSHA SH-27664-SH5 or similar).  
**Features:**
- Tesseract OCR with image preprocessing for handwritten + printed text
- Structured extraction of all permit sections (header, hazards, safety, atmospheric readings, signatures)
- Detection of **non-filled / blank entries** with confidence scoring
- JSON + DataFrame output for downstream integration
- **Annotated image output** with color-coded bounding boxes around every detected word

---

## 1. Setup & Dependencies

In [ ]:
# Install if needed (uncomment):
# !pip install pytesseract Pillow opencv-python-headless pandas

import cv2
import numpy as np
import pytesseract
from PIL import Image, ImageEnhance, ImageFilter
import pandas as pd
import json
import re
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any, Tuple
from enum import Enum
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

print(f"Tesseract version: {pytesseract.get_tesseract_version()}")
print("All dependencies loaded successfully.")

## 2. Core Data Models

In [ ]:
class FillStatus(str, Enum):
    """Status of a form field."""
    FILLED = "filled"
    EMPTY = "empty"
    UNCERTAIN = "uncertain"  # OCR couldn't determine confidently


@dataclass
class FormField:
    """Represents a single key-value pair extracted from the form."""
    key: str
    value: Optional[str]
    status: FillStatus
    confidence: float  # 0.0 - 1.0, OCR confidence for the value
    section: str  # Which section of the permit this belongs to
    raw_text: Optional[str] = None  # Raw OCR output before cleanup

    def to_dict(self) -> dict:
        return {
            "key": self.key,
            "value": self.value,
            "status": self.status.value,
            "confidence": round(self.confidence, 3),
            "section": self.section,
        }


@dataclass
class ExtractionResult:
    """Complete extraction result for one permit."""
    source_file: str
    fields: List[FormField] = field(default_factory=list)
    raw_ocr_text: str = ""
    page_count: int = 0
    ocr_details: Dict[str, Any] = field(default_factory=dict)  # page_name -> {detail_df, image_path}

    @property
    def filled_count(self) -> int:
        return sum(1 for f in self.fields if f.status == FillStatus.FILLED)

    @property
    def empty_count(self) -> int:
        return sum(1 for f in self.fields if f.status == FillStatus.EMPTY)

    @property
    def uncertain_count(self) -> int:
        return sum(1 for f in self.fields if f.status == FillStatus.UNCERTAIN)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([f.to_dict() for f in self.fields])

    def to_json(self) -> str:
        return json.dumps({
            "source_file": self.source_file,
            "page_count": self.page_count,
            "summary": {
                "total_fields": len(self.fields),
                "filled": self.filled_count,
                "empty": self.empty_count,
                "uncertain": self.uncertain_count,
            },
            "fields": [f.to_dict() for f in self.fields],
        }, indent=2)

    def get_empty_fields(self) -> List[FormField]:
        return [f for f in self.fields if f.status == FillStatus.EMPTY]

    def get_uncertain_fields(self) -> List[FormField]:
        return [f for f in self.fields if f.status == FillStatus.UNCERTAIN]


print("Data models defined.")

## 3. Image Preprocessing Pipeline

In [ ]:
class ImagePreprocessor:
    """Preprocessing pipeline optimized for scanned/photographed forms."""

    @staticmethod
    def load_and_preprocess(image_path: str, dpi: int = 300) -> Tuple[np.ndarray, Image.Image]:
        """
        Load an image and apply a full preprocessing pipeline.
        Returns (cv2_image_for_analysis, pil_image_for_ocr).
        """
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Cannot load image: {image_path}")

        # 1. Resize to standard DPI equivalent if too small
        h, w = img.shape[:2]
        if max(h, w) < 2000:
            scale = 2000 / max(h, w)
            img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

        # 2. Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 3. Deskew
        gray = ImagePreprocessor._deskew(gray)

        # 4. Adaptive thresholding for mixed printed + handwritten text
        thresh = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 31, 15
        )

        # 5. Noise removal
        kernel = np.ones((1, 1), np.uint8)
        cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        cleaned = cv2.medianBlur(cleaned, 1)

        # Convert to PIL for pytesseract
        pil_img = Image.fromarray(cleaned)

        # Also enhance the PIL version
        enhancer = ImageEnhance.Contrast(pil_img)
        pil_img = enhancer.enhance(1.5)
        enhancer = ImageEnhance.Sharpness(pil_img)
        pil_img = enhancer.enhance(2.0)

        return cleaned, pil_img

    @staticmethod
    def _deskew(image: np.ndarray) -> np.ndarray:
        """Correct slight rotation in scanned images."""
        coords = np.column_stack(np.where(image < 128))
        if len(coords) < 100:
            return image
        angle = cv2.minAreaRect(coords)[-1]
        if angle < -45:
            angle = -(90 + angle)
        else:
            angle = -angle
        # Only correct small rotations
        if abs(angle) > 10 or abs(angle) < 0.1:
            return image
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)

    @staticmethod
    def prepare_for_checkbox_detection(image_path: str) -> np.ndarray:
        """Prepare image specifically for checkbox / mark detection."""
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        h, w = img.shape[:2]
        if max(h, w) < 2000:
            scale = 2000 / max(h, w)
            img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        return binary


print("Image preprocessor ready.")

## 4. OCR Engine

In [ ]:
class OCREngine:
    """Tesseract OCR wrapper with multiple pass strategies."""

    # Tesseract configs optimized for form fields
    CONFIGS = {
        "default": "--oem 3 --psm 6",          # Uniform block of text
        "single_line": "--oem 3 --psm 7",       # Single line
        "single_word": "--oem 3 --psm 8",       # Single word
        "sparse": "--oem 3 --psm 11",           # Sparse text, no order
        "full_page": "--oem 3 --psm 3",         # Fully auto segmentation
    }

    @staticmethod
    def extract_full_text(pil_image: Image.Image, config: str = "full_page") -> str:
        """Extract all text from the image."""
        return pytesseract.image_to_string(
            pil_image, config=OCREngine.CONFIGS[config]
        )

    @staticmethod
    def extract_with_details(pil_image: Image.Image, config: str = "full_page") -> pd.DataFrame:
        """Extract text with bounding boxes and confidence scores."""
        data = pytesseract.image_to_data(
            pil_image, config=OCREngine.CONFIGS[config],
            output_type=pytesseract.Output.DATAFRAME
        )
        # Filter out empty/low-confidence entries
        data = data.dropna(subset=['text'])
        data = data[data['text'].str.strip() != '']
        return data

    @staticmethod
    def get_line_confidences(detail_df: pd.DataFrame) -> Dict[int, float]:
        """Compute average confidence per text line."""
        if detail_df.empty:
            return {}
        grouped = detail_df.groupby(['block_num', 'par_num', 'line_num'])
        return {
            idx: group['conf'].mean()
            for idx, group in enumerate(grouped)
        }


print("OCR engine ready.")

## 5. Confined Space Permit Parser

This is the core extraction logic, designed around the OSHA Confined Space Entry Permit structure (SH-27664-SH5).

In [ ]:
class ConfinedSpacePermitParser:
    """
    Parses OCR text from a Confined Space Entry Permit into structured fields.
    Handles both Page 1 (permit info, hazards, safety) and Page 2 (atmospheric, signatures).
    """

    # Minimum text length to consider a field "filled"
    MIN_VALUE_LENGTH = 1
    # Confidence threshold below which we mark as "uncertain"
    CONFIDENCE_THRESHOLD = 40.0

    def __init__(self):
        self.fields: List[FormField] = []

    def parse_page1(self, text: str, detail_df: pd.DataFrame) -> List[FormField]:
        """Parse Page 1 of the confined space entry permit."""
        fields = []
        avg_conf = detail_df['conf'].mean() if not detail_df.empty else 0

        # --- HEADER SECTION ---
        header_fields = {
            "Date": r'Date[:\s]*([\d#/]+)',
            "Entry Time": r'Entry\s*Time[:\s]*([\d#:]+)',
            "AM/PM (Entry)": r'Entry\s*Time[:\s]*[\d#:]+\s*(AM|PM)',
            "Permit Expiration Time": r'(?:Permit\s*)?Expiration\s*Time[:\s]*([\d#:]+)',
            "AM/PM (Expiration)": r'Expiration\s*Time[:\s]*[\d#:]+\s*(AM|PM)',
            "Confined Space Name/ID": r'Confined\s*Space\s*Name/?ID[:\s]*(.+?)(?:Location|$)',
            "Location": r'Location[:\s]*(.+?)(?:$|\n)',
            "Reason for Entry": r'Reason\s*for\s*Entry[:\s]*(.+?)(?:$|\n)',
            "Entry Point": r'Entry\s*Point[:\s]*(.+?)(?:Communication|$)',
            "Communication Used": r'Communication\s*used[:\s]*(.+?)(?:$|\n)',
        }

        for key, pattern in header_fields.items():
            fields.append(self._extract_regex(text, key, pattern, "Header", avg_conf))

        # --- ATMOSPHERIC HAZARDS ---
        atmo_hazards = [
            "Oxygen Deficient <19.5%",
            "Oxygen Enriched >= 23.5%",
            "Flammable Gases, Vapors when >= 10% LFL",
            "Toxic Gases, Vapors when >= PEL",
            "Airborne combustible dust",
        ]
        for hazard in atmo_hazards:
            fields.append(self._detect_checkbox(text, hazard, "Atmospheric Hazards", avg_conf))

        atmo_controls = [
            "Test before entry",
            "Continual monitoring",
            "Natural ventilation",
            "Forced air ventilation",
        ]
        for ctrl in atmo_controls:
            fields.append(self._detect_checkbox(text, f"Control: {ctrl}", "Atmospheric Hazards - Controls", avg_conf))

        # --- ENGULFMENT & ENTRAPMENT HAZARDS ---
        engulf_hazards = [
            "Flowing material",
            "Hung up, bridged, crusted material",
            "Inwardly converging walls",
            "Sloping floors",
        ]
        for hazard in engulf_hazards:
            fields.append(self._detect_checkbox(text, hazard, "Engulfment & Entrapment Hazards", avg_conf))

        engulf_controls = [
            "LOTO fill and/or emptying equipment",
            "Lock gates",
            "Block spouts/pipes",
            "Drain/empty",
            "Lifeline use",
        ]
        for ctrl in engulf_controls:
            fields.append(self._detect_checkbox(text, f"Control: {ctrl}", "Engulfment Controls", avg_conf))

        # --- POTENTIAL/KNOWN HAZARDS TABLE ---
        hazard_rows = [
            "Egress hazards", "Insufficient lighting hazard", "Chemical hazards",
            "Mechanical hazards (unguarded items)", "Electrical hazards", "Fall hazards",
            "Respiratory hazards", "Skin hazards", "Heat/Cold hazards",
            "Snake, Rodent, Animal and Insect Hazards", "Vehicle hazards", "Noise hazards",
        ]
        for hazard in hazard_rows:
            fields.append(self._extract_hazard_row(text, hazard, avg_conf))

        fields.append(self._extract_regex(
            text, "Other Hazards & Control",
            r'Other\s*Hazards\s*&?\s*Control[:\s]*(.+?)(?:Safety|$)',
            "Potential/Known Hazards", avg_conf
        ))

        # --- SAFETY & EMERGENCY RESCUE ---
        safety_items = [
            "Entry area secure",
            "LOTO/de-energization & isolation",
            "Lighting (rated for type of space/work)",
            "Hot work permit",
            "GFCI equipment",
            "Non-sparking tools",
            "Safety harness & lifeline or retrieval line",
            "PPE inspection completed before use",
            "Mechanical retrieval device",
            "Respirator",
            "Hearing Protection",
        ]
        for item in safety_items:
            fields.append(self._detect_yes_no_na(text, item, "Safety & Emergency Rescue", avg_conf))

        fields.append(self._extract_regex(
            text, "Other PPE",
            r'Other\s*PPE[:\s]*(.+?)(?:Entrants|$)',
            "Safety & Emergency Rescue", avg_conf
        ))

        # --- RESCUE INFO ---
        rescue_fields = {
            "Rescue equipment available": r'Rescue\s*equipment\s*available[?]?\s*(YES|NO)',
            "Rescue equipment type": r'Type[:\s]*(.+?)(?:$|\n)',
            "Stand by personnel used": r'Stand\s*by\s*personnel\s*used[?]?\s*(YES|NO)',
            "Stand by personnel names": r'Stand\s*by.*?Name\(?s?\)?[:\s]*(.+?)(?:$|\n)',
            "CPR trained person available": r'CPR\s*trained\s*person\s*available[?]?\s*(YES|NO)',
            "CPR trained person names": r'CPR.*?Name\(?s?\)?[:\s]*(.+?)(?:$|\n)',
        }
        for key, pattern in rescue_fields.items():
            fields.append(self._extract_regex(text, key, pattern, "Rescue Information", avg_conf))

        return fields

    def parse_page2(self, text: str, detail_df: pd.DataFrame) -> List[FormField]:
        """Parse Page 2: Atmospheric info, air monitoring, signatures."""
        fields = []
        avg_conf = detail_df['conf'].mean() if not detail_df.empty else 0

        # --- ATMOSPHERIC INFORMATION ---
        atmo_fields = {
            "Monitor calibrated": r'Monitor\s*calibrated[?]?\s*(YES|NO)',
            "Date calibrated": r'Date\s*calibrated[:\s]*([\d#/]+)',
            "Monitor functioning correctly": r'Monitor\s*functioning\s*correctly[?]?\s*(YES|NO)',
        }
        for key, pattern in atmo_fields.items():
            fields.append(self._extract_regex(text, key, pattern, "Atmospheric Information", avg_conf))

        # --- PRE-ENTRY AIR MONITOR READINGS ---
        pre_entry_fields = {
            "Pre-entry Time": r'Pre[\-\s]*entry.*?Time\s+([\d#:]+)',
            "Pre-entry O2": r'(?:Pre.*?|Initial.*?)O2.*?([\d.]+)',
            "Pre-entry LFL": r'LFL.*?([\d.]+)',
            "Pre-entry CO": r'CO.*?([\d.]+)',
            "Pre-entry H2S": r'H2S.*?([\d.]+)',
            "Pre-entry PH3": r'PH3.*?([\d.]+)',
            "Pre-entry NH3": r'NH3.*?([\d.]+)',
            "Pre-entry Signature": r'Pre.*?Signature[:\s]*(.+?)(?:$|\n)',
        }
        for key, pattern in pre_entry_fields.items():
            fields.append(self._extract_regex(text, key, pattern, "Pre-Entry Air Monitor Readings", avg_conf))

        # --- PERIODIC AIR MONITOR READINGS (detect rows) ---
        periodic_rows = self._extract_periodic_readings(text, avg_conf)
        fields.extend(periodic_rows)

        # --- SIGNATURES ---
        sig_fields = {
            "Entrant(s)": r'Entrant\(?s?\)?[:\s]*(.+?)(?:$|\n)',
            "Attendant(s)": r'Attendant\(?s?\)?[:\s]*(.+?)(?:$|\n)',
            "Entry Supervisor (Print)": r'Entry\s*Supervisor[:\s]*(.+?)(?:Signature|$)',
            "Entry Supervisor (Signature)": r'Entry\s*Supervisor.*?Signature[:\s]*(.+?)(?:$|\n)',
        }
        for key, pattern in sig_fields.items():
            fields.append(self._extract_regex(text, key, pattern, "Signatures & Authorization", avg_conf))

        # --- PERMIT CANCELLATION ---
        cancel_fields = {
            "Permit Cancellation Time": r'Permit\s*Cancellation\s*Time[:\s]*([\d#:]+)',
            "Structure returned to operating condition": r'Structure\s*returned.*?\s*(YES|NO)',
            "Cancellation Date": r'Date[:\s]*([\d#/]+)\s*$',
        }
        for key, pattern in cancel_fields.items():
            fields.append(self._extract_regex(text, key, pattern, "Permit Cancellation", avg_conf))

        return fields

    # ── Helper methods ──────────────────────────────────────────────

    def _extract_regex(self, text: str, key: str, pattern: str,
                       section: str, avg_conf: float) -> FormField:
        """Extract a value using regex; determine fill status."""
        match = re.search(pattern, text, re.IGNORECASE | re.MULTILINE)
        if match:
            raw_value = match.group(1).strip()
            cleaned = self._clean_value(raw_value)
            status = self._determine_status(cleaned, avg_conf)
            return FormField(
                key=key, value=cleaned if status != FillStatus.EMPTY else None,
                status=status, confidence=avg_conf / 100.0,
                section=section, raw_text=raw_value
            )
        return FormField(
            key=key, value=None, status=FillStatus.EMPTY,
            confidence=0.0, section=section
        )

    def _detect_checkbox(self, text: str, label: str,
                         section: str, avg_conf: float) -> FormField:
        """Detect if a checkbox item appears in the text (indicating it was checked)."""
        # Simplify label for fuzzy matching
        label_words = label.lower().split()
        text_lower = text.lower()
        # Check if key words from the label appear near each other
        found = all(w in text_lower for w in label_words[:3])  # first 3 words
        return FormField(
            key=label,
            value="Checked (present in form)" if found else None,
            status=FillStatus.FILLED if found else FillStatus.UNCERTAIN,
            confidence=avg_conf / 100.0 if found else 0.3,
            section=section
        )

    def _detect_yes_no_na(self, text: str, label: str,
                          section: str, avg_conf: float) -> FormField:
        """Detect YES/NO/NA for safety checklist items."""
        label_words = label.lower().split()[:3]
        text_lower = text.lower()
        found = all(w in text_lower for w in label_words)
        return FormField(
            key=label,
            value="Present in form" if found else None,
            status=FillStatus.FILLED if found else FillStatus.UNCERTAIN,
            confidence=avg_conf / 100.0 if found else 0.3,
            section=section
        )

    def _extract_hazard_row(self, text: str, hazard_name: str,
                            avg_conf: float) -> FormField:
        """Extract a row from the Potential/Known Hazards table."""
        keywords = hazard_name.lower().split()[:2]
        text_lower = text.lower()
        found = all(w in text_lower for w in keywords)
        return FormField(
            key=f"Hazard: {hazard_name}",
            value="Listed in form" if found else None,
            status=FillStatus.FILLED if found else FillStatus.UNCERTAIN,
            confidence=avg_conf / 100.0 if found else 0.3,
            section="Potential/Known Hazards"
        )

    def _extract_periodic_readings(self, text: str, avg_conf: float) -> List[FormField]:
        """Extract periodic air monitoring reading rows."""
        fields = []
        # Look for lines with time patterns followed by numeric readings
        pattern = r'([\d#:]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)'
        matches = re.finditer(pattern, text)
        row_num = 0
        for match in matches:
            row_num += 1
            reading = {
                "time": match.group(1),
                "O2": match.group(2),
                "LFL": match.group(3),
                "CO": match.group(4),
                "H2S": match.group(5),
                "PH3": match.group(6),
                "NH3": match.group(7),
            }
            fields.append(FormField(
                key=f"Periodic Reading Row {row_num}",
                value=json.dumps(reading),
                status=FillStatus.FILLED,
                confidence=avg_conf / 100.0,
                section="Periodic Air Monitor Readings"
            ))

        if row_num == 0:
            fields.append(FormField(
                key="Periodic Air Monitor Readings",
                value=None,
                status=FillStatus.EMPTY,
                confidence=0.0,
                section="Periodic Air Monitor Readings"
            ))

        return fields

    def _clean_value(self, value: str) -> str:
        """Clean up OCR artifacts from extracted values."""
        # Remove common OCR noise
        value = re.sub(r'[|\\{}\[\]]', '', value)
        value = re.sub(r'\s+', ' ', value).strip()
        # Remove trailing punctuation artifacts
        value = value.strip('_-=.')
        return value

    def _determine_status(self, value: str, avg_conf: float) -> FillStatus:
        """Determine if a field is filled, empty, or uncertain."""
        if not value or len(value.strip()) < self.MIN_VALUE_LENGTH:
            return FillStatus.EMPTY
        # If value is all placeholder chars (#, _, -), it's a template marker
        if re.match(r'^[#_\-\s.]+$', value):
            return FillStatus.EMPTY
        if avg_conf < self.CONFIDENCE_THRESHOLD:
            return FillStatus.UNCERTAIN
        return FillStatus.FILLED


print("Permit parser ready.")

## 6. Main Extraction Pipeline

In [ ]:
class PermitExtractor:
    """
    End-to-end pipeline: Image → Preprocess → OCR → Parse → Structured Output.
    Supports single images or multi-page permits (pass page 1 and page 2 separately).
    """

    def __init__(self):
        self.preprocessor = ImagePreprocessor()
        self.ocr = OCREngine()
        self.parser = ConfinedSpacePermitParser()

    def extract_permit(self, page1_path: str, page2_path: str = None) -> ExtractionResult:
        """
        Extract all key-value pairs from a confined space entry permit.

        Args:
            page1_path: Path to Page 1 image
            page2_path: Path to Page 2 image (optional)

        Returns:
            ExtractionResult with all fields, statuses, and confidence scores
        """
        result = ExtractionResult(source_file=page1_path)
        all_raw_text = ""

        # --- Process Page 1 ---
        print("Processing Page 1...")
        cv_img1, pil_img1 = self.preprocessor.load_and_preprocess(page1_path)
        text1 = self.ocr.extract_full_text(pil_img1)
        detail1 = self.ocr.extract_with_details(pil_img1)
        all_raw_text += f"=== PAGE 1 ===\n{text1}\n"
        result.page_count = 1

        page1_fields = self.parser.parse_page1(text1, detail1)
        result.fields.extend(page1_fields)
        result.ocr_details['page1'] = {'detail_df': detail1, 'image_path': page1_path, 'pil_img': pil_img1}
        print(f"  → Extracted {len(page1_fields)} fields from Page 1")

        # --- Process Page 2 ---
        if page2_path and os.path.exists(page2_path):
            print("Processing Page 2...")
            cv_img2, pil_img2 = self.preprocessor.load_and_preprocess(page2_path)
            text2 = self.ocr.extract_full_text(pil_img2)
            detail2 = self.ocr.extract_with_details(pil_img2)
            all_raw_text += f"\n=== PAGE 2 ===\n{text2}\n"
            result.page_count = 2

            page2_fields = self.parser.parse_page2(text2, detail2)
            result.fields.extend(page2_fields)
            result.ocr_details['page2'] = {'detail_df': detail2, 'image_path': page2_path, 'pil_img': pil_img2}
            print(f"  → Extracted {len(page2_fields)} fields from Page 2")

        result.raw_ocr_text = all_raw_text

        print(f"\nExtraction complete: {len(result.fields)} total fields")
        print(f"  Filled:    {result.filled_count}")
        print(f"  Empty:     {result.empty_count}")
        print(f"  Uncertain: {result.uncertain_count}")

        return result


print("Extraction pipeline ready.")

## 7. Run Extraction on Sample Permit

In [ ]:
# ── Configure paths ──
PAGE1_PATH = "/mnt/user-data/uploads/IMG_7043.jpg"  # Page 1 of the permit
PAGE2_PATH = "/mnt/user-data/uploads/IMG_7044.jpg"  # Page 2 of the permit

# ── Run extraction ──
extractor = PermitExtractor()
result = extractor.extract_permit(PAGE1_PATH, PAGE2_PATH)

## 8. View Results

In [ ]:
# ── Full results as DataFrame ──
df = result.to_dataframe()
print(f"Total fields extracted: {len(df)}")
print(f"\nStatus breakdown:")
print(df['status'].value_counts().to_string())
print("\n" + "="*80)
df.style.map(
    lambda v: 'background-color: #ffcccc' if v == 'empty'
    else ('background-color: #ffffcc' if v == 'uncertain'
          else 'background-color: #ccffcc'),
    subset=['status']
)

In [ ]:
# ── Non-filled entries (EMPTY + UNCERTAIN) ──
print("⚠️  NON-FILLED OR UNCERTAIN ENTRIES:")
print("=" * 60)
empty_fields = result.get_empty_fields()
uncertain_fields = result.get_uncertain_fields()

if empty_fields:
    print(f"\n🔴 EMPTY ({len(empty_fields)} fields):")
    for f in empty_fields:
        print(f"   [{f.section}] {f.key}")

if uncertain_fields:
    print(f"\n🟡 UNCERTAIN ({len(uncertain_fields)} fields):")
    for f in uncertain_fields:
        print(f"   [{f.section}] {f.key} (conf: {f.confidence:.1%})")

if not empty_fields and not uncertain_fields:
    print("\n✅ All fields appear to be filled.")

In [ ]:
# ── Filled entries with values ──
print("✅ FILLED ENTRIES:")
print("=" * 60)
filled = df[df['status'] == 'filled']
for _, row in filled.iterrows():
    val_display = row['value'][:80] + '...' if row['value'] and len(row['value']) > 80 else row['value']
    print(f"   [{row['section']}] {row['key']}: {val_display}")

In [ ]:
# ── Export as JSON ──
json_output = result.to_json()
print(json_output[:3000])  # Preview first 3000 chars

# Save to file
with open("permit_extraction.json", "w") as f:
    f.write(json_output)
print(f"\n→ Saved full JSON to permit_extraction.json")

## 9. Annotated Image — Bounding Box Visualization

Generates annotated copies of the original permit images with **color-coded bounding boxes** around every detected word:
- **Green** = High confidence (≥ 70%)
- **Orange** = Medium confidence (40–69%)
- **Red** = Low confidence (< 40%)

Outputs are saved as PNG files alongside the notebook.

In [ ]:
class PermitVisualizer:
    """
    Draw color-coded bounding boxes on the original permit images
    based on Tesseract OCR word-level detections.
    """

    # Color thresholds and corresponding colors (BGR for OpenCV)
    HIGH_CONF = 70   # green
    MED_CONF  = 40   # orange
    # below MED_CONF  → red

    COLOR_HIGH   = (0, 200, 0)     # green BGR
    COLOR_MED    = (0, 165, 255)   # orange BGR
    COLOR_LOW    = (0, 0, 220)     # red BGR

    # Matplotlib equivalents (RGB normalized)
    MPL_HIGH  = (0.0, 0.78, 0.0, 0.45)
    MPL_MED   = (1.0, 0.65, 0.0, 0.45)
    MPL_LOW   = (0.86, 0.0, 0.0, 0.45)

    EDGE_HIGH = (0.0, 0.78, 0.0, 1.0)
    EDGE_MED  = (1.0, 0.65, 0.0, 1.0)
    EDGE_LOW  = (0.86, 0.0, 0.0, 1.0)

    @classmethod
    def _conf_color_mpl(cls, conf: float):
        """Return (face_color, edge_color) for matplotlib based on confidence."""
        if conf >= cls.HIGH_CONF:
            return cls.MPL_HIGH, cls.EDGE_HIGH
        elif conf >= cls.MED_CONF:
            return cls.MPL_MED, cls.EDGE_MED
        else:
            return cls.MPL_LOW, cls.EDGE_LOW

    @classmethod
    def _conf_color_cv(cls, conf: float):
        """Return BGR color for OpenCV based on confidence."""
        if conf >= cls.HIGH_CONF:
            return cls.COLOR_HIGH
        elif conf >= cls.MED_CONF:
            return cls.COLOR_MED
        else:
            return cls.COLOR_LOW

    @classmethod
    def annotate_image_cv(cls, image_path: str, detail_df: pd.DataFrame,
                          output_path: str = None) -> np.ndarray:
        """
        Draw bounding boxes on the image using OpenCV and save to disk.

        Args:
            image_path: Path to the original (or preprocessed) image
            detail_df: Tesseract detail DataFrame (from image_to_data)
            output_path: Where to save the annotated image

        Returns:
            Annotated image as numpy array (BGR)
        """
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Cannot read: {image_path}")

        # Scale factor: detail_df coords are relative to the preprocessed image size.
        # We draw on the original so we need to match dimensions.
        # The preprocessor may have resized, so we scale boxes to original.
        orig_h, orig_w = img.shape[:2]

        # Get the preprocessed image size from the detail_df bounding boxes
        if detail_df.empty:
            print(f"  ⚠ No OCR words detected for {image_path}")
            return img

        max_x = (detail_df['left'] + detail_df['width']).max()
        max_y = (detail_df['top'] + detail_df['height']).max()
        scale_x = orig_w / max(max_x, 1)
        scale_y = orig_h / max(max_y, 1)

        overlay = img.copy()
        word_count = 0

        for _, row in detail_df.iterrows():
            conf = float(row['conf'])
            text = str(row['text']).strip()
            if not text:
                continue

            x = int(row['left'] * scale_x)
            y = int(row['top'] * scale_y)
            w = int(row['width'] * scale_x)
            h = int(row['height'] * scale_y)

            color = cls._conf_color_cv(conf)

            # Semi-transparent filled rectangle
            cv2.rectangle(overlay, (x, y), (x + w, y + h), color, -1)
            # Solid border
            cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
            word_count += 1

        # Blend overlay for transparency
        alpha = 0.25
        annotated = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

        # Add legend in top-left corner
        legend_y = 30
        for label, color in [('High conf (>=70%)', cls.COLOR_HIGH),
                              ('Med conf (40-69%)', cls.COLOR_MED),
                              ('Low conf (<40%)', cls.COLOR_LOW)]:
            cv2.rectangle(annotated, (10, legend_y - 18), (30, legend_y), color, -1)
            cv2.rectangle(annotated, (10, legend_y - 18), (30, legend_y), (0,0,0), 1)
            cv2.putText(annotated, label, (38, legend_y - 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            legend_y += 30

        if output_path:
            cv2.imwrite(output_path, annotated)
            print(f"  → Saved annotated image: {output_path} ({word_count} words boxed)")

        return annotated

    @classmethod
    def annotate_image_matplotlib(cls, image_path: str, detail_df: pd.DataFrame,
                                  output_path: str = None,
                                  title: str = 'OCR Bounding Boxes') -> None:
        """
        Higher-quality annotated image using matplotlib (good for notebooks).
        Shows the original image with semi-transparent colored rectangles.
        """
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Cannot read: {image_path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = img.shape[:2]

        if detail_df.empty:
            print(f"  ⚠ No OCR words detected for {image_path}")
            return

        max_x = (detail_df['left'] + detail_df['width']).max()
        max_y = (detail_df['top'] + detail_df['height']).max()
        scale_x = orig_w / max(max_x, 1)
        scale_y = orig_h / max(max_y, 1)

        fig_w = 16
        fig_h = fig_w * (orig_h / orig_w)
        fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h))
        ax.imshow(img_rgb)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')

        counts = {'high': 0, 'med': 0, 'low': 0}

        for _, row in detail_df.iterrows():
            conf = float(row['conf'])
            text = str(row['text']).strip()
            if not text:
                continue

            x = row['left'] * scale_x
            y = row['top'] * scale_y
            w = row['width'] * scale_x
            h = row['height'] * scale_y

            face, edge = cls._conf_color_mpl(conf)
            rect = patches.Rectangle((x, y), w, h,
                                     linewidth=1.5, edgecolor=edge,
                                     facecolor=face)
            ax.add_patch(rect)

            if conf >= cls.HIGH_CONF:
                counts['high'] += 1
            elif conf >= cls.MED_CONF:
                counts['med'] += 1
            else:
                counts['low'] += 1

        # Legend
        legend_elements = [
            patches.Patch(facecolor=cls.MPL_HIGH, edgecolor=cls.EDGE_HIGH,
                          label=f'High ≥70% ({counts["high"]} words)'),
            patches.Patch(facecolor=cls.MPL_MED, edgecolor=cls.EDGE_MED,
                          label=f'Med 40-69% ({counts["med"]} words)'),
            patches.Patch(facecolor=cls.MPL_LOW, edgecolor=cls.EDGE_LOW,
                          label=f'Low <40% ({counts["low"]} words)'),
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=11,
                  framealpha=0.9, edgecolor='black')

        plt.tight_layout()
        if output_path:
            fig.savefig(output_path, dpi=150, bbox_inches='tight',
                        facecolor='white', edgecolor='none')
            print(f"  → Saved: {output_path}")
        plt.show()
        plt.close(fig)


print("Visualizer ready.")

In [ ]:
# ── Generate annotated images for each page ──
viz = PermitVisualizer()

for page_key, page_label in [('page1', 'Page 1 — Permit Info & Hazards'),
                              ('page2', 'Page 2 — Atmospheric & Signatures')]:
    if page_key in result.ocr_details:
        info = result.ocr_details[page_key]
        print(f"\nAnnotating {page_label}...")

        # High-quality matplotlib version (displays in notebook)
        mpl_path = f"annotated_{page_key}_matplotlib.png"
        viz.annotate_image_matplotlib(
            info['image_path'], info['detail_df'],
            output_path=mpl_path, title=page_label
        )

        # OpenCV version (lighter, good for pipelines)
        cv_path = f"annotated_{page_key}_cv.png"
        viz.annotate_image_cv(
            info['image_path'], info['detail_df'],
            output_path=cv_path
        )

print("\n✅ All annotated images generated.")

## 10. View Raw OCR Output (Debug)

In [ ]:
# ── Raw OCR text for debugging / manual review ──
print(result.raw_ocr_text)

## 11. Batch Processing (Multiple Permits)

Use this section to process a folder of permits.

In [ ]:
def batch_extract(permit_pairs: List[Dict[str, str]], output_dir: str = "./results") -> pd.DataFrame:
    """
    Process multiple permits in batch.

    Args:
        permit_pairs: List of dicts with 'page1' and optional 'page2' keys
            Example: [{'page1': 'img1.jpg', 'page2': 'img2.jpg'}, ...]
        output_dir: Directory to save individual JSON results

    Returns:
        Summary DataFrame of all permits
    """
    os.makedirs(output_dir, exist_ok=True)
    extractor = PermitExtractor()
    summary_rows = []

    for i, pair in enumerate(permit_pairs):
        permit_id = f"permit_{i+1:03d}"
        print(f"\n{'='*60}")
        print(f"Processing {permit_id}: {pair['page1']}")
        print(f"{'='*60}")

        try:
            result = extractor.extract_permit(
                pair['page1'],
                pair.get('page2')
            )

            # Save JSON
            json_path = os.path.join(output_dir, f"{permit_id}.json")
            with open(json_path, 'w') as f:
                f.write(result.to_json())

            summary_rows.append({
                "permit_id": permit_id,
                "source": pair['page1'],
                "total_fields": len(result.fields),
                "filled": result.filled_count,
                "empty": result.empty_count,
                "uncertain": result.uncertain_count,
                "completion_pct": f"{result.filled_count / max(len(result.fields), 1) * 100:.1f}%",
                "status": "OK",
            })
        except Exception as e:
            summary_rows.append({
                "permit_id": permit_id,
                "source": pair['page1'],
                "total_fields": 0,
                "filled": 0,
                "empty": 0,
                "uncertain": 0,
                "completion_pct": "N/A",
                "status": f"ERROR: {str(e)[:60]}",
            })

    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(output_dir, "batch_summary.csv"), index=False)
    print(f"\n\nBatch complete. Results saved to {output_dir}/")
    return summary_df


# Example usage (uncomment to run):
# permits = [
#     {"page1": "/path/to/permit1_pg1.jpg", "page2": "/path/to/permit1_pg2.jpg"},
#     {"page1": "/path/to/permit2_pg1.jpg"},
# ]
# summary = batch_extract(permits, output_dir="./permit_results")
# display(summary)

print("Batch processor ready.")